# Report: distribution of GRM relatedness per sample set
Off-diagonal GRM entries from each run's `grm_shards`, binned; share of that run's pairs per bin. Runs without a finished GRM are skipped. Bins with 1-20 pairs are not drawn or written. Figures and the masked table go to `report/grm_dist` of `RUN_ID`.

In [ ]:
RUN_ID = "chb_chs"   # only sets up the session and where the report goes

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
RUN_IDS = ["chb_chs", "yri", "lwk", "stu_itu_beb", "pjl", "asw", "mxl", "uniform_10k", "even_10k"]
EDGES = np.round(np.arange(-0.2, 1.2005, 0.001), 4)

In [ ]:
def offdiag_hist(shards, n, edges):
    """Histogram of the off-diagonal entries, shard by shard; shards concatenate to the lower triangle."""
    counts, off = np.zeros(len(edges) - 1, np.int64), 0
    for s in shards:
        v = np.fromfile(s, dtype="float32")
        k = np.arange(1, n + 1, dtype=np.int64)
        d = k * (k + 1) // 2 - 1 - off            # diagonal positions inside this shard
        d = d[(d >= 0) & (d < len(v))]
        keep = np.ones(len(v), bool); keep[d] = False
        counts += np.histogram(v[keep], bins=edges)[0]
        off += len(v)
    assert off == n * (n + 1) // 2, (off, n)
    return counts

rows = {}
for run_id in RUN_IDS:
    c = load_run(run_id)
    if not stage_status(c, "grm_shards").done:
        print(run_id, "no finished GRM, skipped"); continue
    shards = [u for u, _ in target_items(c, "grm_shards")]
    n = sum(1 for _ in open(shards[0].split(".grm.bin.")[0] + ".grm.id"))
    rows[run_id] = (n, offdiag_hist(shards, n, EDGES))
    print(run_id, f"{n:,} people, {n * (n - 1) // 2:,} pairs")

In [ ]:
OUTDIR = os.path.expanduser(cfg.path("report", "grm_dist"))
os.makedirs(OUTDIR, exist_ok=True)
mid = (EDGES[:-1] + EDGES[1:]) / 2
tab = pd.DataFrame({"bin_mid": mid})
for r, (n, cnt) in rows.items():
    pairs = n * (n - 1) // 2
    tab[f"{r}_share"] = np.where(cnt >= 21, cnt / pairs, np.nan)   # 1-20 pairs: not shown
tab.dropna(how="all", subset=tab.columns[1:]).to_csv(f"{OUTDIR}/grm_offdiag_share.tsv", sep="\t", index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for r, (n, _) in rows.items():
    y = tab[f"{r}_share"]
    axes[0].plot(mid, y, lw=1.2, label=f"{r} (N={n:,})")
    axes[1].plot(mid, y, lw=1, label=r)
axes[0].set(xlim=(-0.06, 0.06), xlabel="GRM (off-diagonal)", ylabel="share of pairs per 0.001 bin", title="bulk")
axes[1].set(yscale="log", xlim=(-0.1, 1.1), xlabel="GRM (off-diagonal)", title="all, log scale (relatives on the right)")
axes[0].legend(fontsize=8)
plt.tight_layout()
fig.savefig(f"{OUTDIR}/grm_offdiag_share.png", dpi=130, bbox_inches="tight"); plt.show()

summ = pd.DataFrame([{"run": r, "n": n, "pairs": n * (n - 1) // 2,
                      "sd": float(np.sqrt(np.sum(cnt * mid ** 2) / cnt.sum() - (np.sum(cnt * mid) / cnt.sum()) ** 2)),
                      **{f"share_above_{t}": float(cnt[mid > t].sum() / cnt.sum()) for t in (0.05, 0.1, 0.2, 0.4)}}
                     for r, (n, cnt) in rows.items()])
for t in (0.05, 0.1, 0.2, 0.4):   # mask shares that rest on 1-20 pairs
    above = np.array([cnt[mid > t].sum() for _, cnt in rows.values()])
    summ[f"share_above_{t}"] = summ[f"share_above_{t}"].where(~((above >= 1) & (above <= 20)))
summ.to_csv(f"{OUTDIR}/grm_summary.tsv", sep="\t", index=False)
json.dump({"run_id": "grm_dist", "stage": "grm_dist", "notes": {"runs": list(rows), "bin": 0.001}, "checks": [],
           "files": [{"file": "grm_offdiag_share.png", "kind": "figure", "caption": "share of each run's pairs per bin; bins with 1-20 pairs not drawn"},
                     {"file": "grm_summary.tsv", "kind": "table", "caption": "SD of off-diagonal GRM, share of pairs above thresholds"}]},
          open(f"{OUTDIR}/summary.json", "w"), indent=2)
summ